# P0：RQ2 下游出发延误预测与解释

研究问题：上游飞机实际到达后，能否预测下游出发延误？P0 使用三个朴素基线与计划信息 G0 线性回归。此页从该轮完整预测与指标归档读取结果，不重新训练。

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image, Markdown

if "ROOT" not in globals():
    ROOT = Path.cwd()
    while not (ROOT / "scripts/build_features.py").exists():
        if ROOT.parent == ROOT:
            raise RuntimeError("请在 Airline-delay 仓库中运行 Notebook")
        ROOT = ROOT.parent
if "RUN_DIR" not in globals():
    candidates = [p for p in (ROOT / "experiments/runs").glob("R*")
                  if (p / "metrics.csv").exists() and (p / "eda_summary.json").exists()
                  and (p / "rq1_summary.json").exists()]
    if not candidates:
        raise RuntimeError("请先运行 P0 开发分析，或手动设置 RUN_DIR")
    RUN_DIR = max(candidates, key=lambda p: int(p.name[1:]))
TABLES = RUN_DIR / "tables"
FIGURES = RUN_DIR / "figures"
def table(name):
    return pd.read_csv(TABLES / name)
def figure(name):
    display(Image(filename=str(FIGURES / name)))
def receipt(name):
    return json.loads((RUN_DIR / name).read_text(encoding="utf-8"))
display(Markdown(f"**归档运行：{RUN_DIR.name}**。本 Notebook 读取该轮表格与图，不重新拟合模型。"))

## 1. 预测时点、时间折与标签成熟

固定 UTC 前向验证：4 月训练→5 月验证；4–5 月训练→6 月验证；4–6 月训练→7 月验证。预测时间窗为半开区间。训练记录不仅预测时点须早于截止，实际出发标签也须严格早于该折截止；在月初同刻才成熟的标签不能用于训练。8 月不参与本轮开发。

In [ ]:
splits = pd.read_csv(RUN_DIR / "split_audit.csv")
display(splits)
model_manifest = receipt("model_manifest.json")
display(pd.DataFrame([{k: model_manifest[k] for k in ["evaluation_rows", "evaluation_ids_sha256", "split_sha256"]}]))

## 2. G0 特征和朴素基线

G0 仅包含机场、航线、航司、航程、计划飞行时长、星期、计划出发小时、计划周转。上游到达延误仅用于本轮 carry-forward 业务基线，还未加入 G0。实际地面时间、下游实际出发和结果字段禁止入模。训练均值/中位数在每折训练段估计；填补、类别合并、编码和缩放均在该训练段拟合。

In [ ]:
display(pd.DataFrame({"G0特征": model_manifest["features"]}))
display(pd.DataFrame([model_manifest["effective_preprocessing"]]))

## 3. 每折与加权整体误差

主要指标为 MAE，RMSE 反映大误差的影响。所有模型统一先将负预测截断为零。整体 MAE 使用所有验证记录的绝对误差总量除以总样本数，整体 RMSE 也由平方误差总量计算；不是三个折指标的简单平均。以下结果只用于建立初始参照。

In [ ]:
metrics = pd.read_csv(RUN_DIR / "metrics.csv")
main = metrics.loc[metrics["group"].eq("all")]
display(main[["model", "fold", "n", "mae", "rmse", "target_ge15_share"]])
figure("rq2_metric_comparison.png")
pooled = main.loc[main.fold.eq("overall")].set_index("model")
comparison = pooled[["n", "mae", "rmse"]].copy()
comparison["MAE改善_vs_carry_forward_分钟"] = pooled.loc["carry_forward", "mae"] - comparison.mae
display(comparison)

## 4. 严重延误、隔夜及运行分组

普通/严重组按真实非负目标预定义为 <60 / ≥60 分钟。这些是结果人群的误差分层，不能作为训练后筛选有利人群的方法。隔夜、机场、航司、星期和计划小时同时给出样本数；小样本组误差应谨慎解读。

In [ ]:
overall = metrics.loc[metrics.fold.eq("overall")]
for group in ["severity", "overnight", "day_of_week", "scheduled_hour"]:
    display(Markdown(f"**{group}**"))
    display(overall.loc[overall["group"].eq(group), ["model", "value", "n", "mae", "rmse"]])
for group in ["airport", "airline"]:
    display(Markdown(f"**{group}（G0样本最多20组）**"))
    display(overall.loc[overall["group"].eq(group) & overall.model.eq("g0_linear"), ["value", "n", "mae", "rmse"]].sort_values("n", ascending=False).head(20))

## 5. 下一轮比较与证据

每折完整预测在 predictions/，模型在 models/，实际评估 connection_id 在 evaluation_ids.csv.gz。后续添加 G1、机型、机场运行与天气时使用同一评估 ID、相同时间折和线性模型设置，逐轮只改变一项研究因素。当前没有天气增益、复杂模型增益或最终 8 月测试结论。